# Student Placement Prediction — Logistic Regression

In [1]:
import pandas as pd
import numpy as np
import word2number as w2n

df = pd.read_csv("student_dirty_dataset.csv")

print("=== Raw Data ===")
print(df)
print("\nShape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

=== Raw Data ===
    student_id     name         age  cgpa attendance internship placed
0          101  Krishna          21   8.5         92          1      1
1          102    Aman           22   7.2         85          0      0
2          103    Priya         NaN   9.1         96          1      1
3          104    Rahul          20   NaN         75          0      0
4          105  Sneha            23   8.0        NaN          1      1
5          106    Arjun  twenty two   7.5         88        NaN      0
6          107    Pooja          21  nine         91          1    NaN
7          108   Vikram         NaN   6.8     eighty          0      0
8          109     Neha          20   8.7         89        yes      1
9          110    Kiran          22   7.0         79          0     No
10         111     Riya          23   NaN         95          1      1
11         112      Dev          21   8.2        NaN        NaN    NaN
12         103    Priya         NaN   9.1         96        

# Data Cleaning

In [2]:
def convert_to_number(x):
    """Convert a value to float. Handles word-numbers like 'twenty two' via word2number."""
    try:
        return float(x)
    except:
        try:
            return float(w2n.word_to_num(str(x)))
        except:
            return None

# Step 1: Strip whitespace from name column
df["name"] = df["name"].str.strip()

# Step 2: Drop duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Step 3: Replace text labels with numeric values BEFORE fillna
#         Must come before fillna so 'yes'/'No' don't get overwritten by 0 first
df["internship"] = df["internship"].replace({"yes": 1, "Yes": 1})
df["placed"]     = df["placed"].replace({"No": 0, "no": 0})

# Step 4: Convert all text/word numbers to numeric (includes age, cgpa, attendance)
#         BUG FIX: age conversion was originally commented out
df["age"]        = df["age"].apply(convert_to_number)
df["cgpa"]       = df["cgpa"].apply(convert_to_number)
df["attendance"] = df["attendance"].apply(convert_to_number)
df["internship"] = df["internship"].apply(convert_to_number)
df["placed"]     = df["placed"].apply(convert_to_number)

# Step 5: Fill remaining NaN values with column mean / 0
df["age"]        = df["age"].fillna(df["age"].mean())
df["cgpa"]       = df["cgpa"].fillna(df["cgpa"].mean())
df["attendance"] = df["attendance"].fillna(df["attendance"].mean())
df["internship"] = df["internship"].fillna(0)
df["placed"]     = df["placed"].fillna(0)

# Step 6: Cast binary columns to int
df["internship"] = df["internship"].astype(int)
df["placed"]     = df["placed"].astype(int)

print("=== Cleaned Data ===")
print(df)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

=== Cleaned Data ===
    student_id     name        age      cgpa  attendance  internship  placed
0          101  Krishna  21.000000  8.500000   92.000000           1       1
1          102     Aman  22.000000  7.200000   85.000000           0       0
2          103    Priya  21.444444  9.100000   96.000000           1       1
3          104    Rahul  20.000000  7.888889   75.000000           0       0
4          105    Sneha  23.000000  8.000000   87.777778           1       1
5          106    Arjun  21.444444  7.500000   88.000000           0       0
6          107    Pooja  21.000000  7.888889   91.000000           1       0
7          108   Vikram  21.444444  6.800000   87.777778           0       0
8          109     Neha  20.000000  8.700000   89.000000           1       1
9          110    Kiran  22.000000  7.000000   79.000000           0       0
10         111     Riya  23.000000  7.888889   95.000000           1       1
11         112      Dev  21.000000  8.200000   87.77777

# Verify Clean Data

In [3]:
print("=== First 5 Rows ===")
print(df.head())

print("\n=== Unique values in 'placed' ===")
print(df["placed"].unique(), "| dtype:", df["placed"].dtype)

print("\n=== Unique values in 'internship' ===")
print(df["internship"].unique(), "| dtype:", df["internship"].dtype)

print("\n=== Target Distribution ===")
print(df["placed"].value_counts())

=== First 5 Rows ===
   student_id     name        age      cgpa  attendance  internship  placed
0         101  Krishna  21.000000  8.500000   92.000000           1       1
1         102     Aman  22.000000  7.200000   85.000000           0       0
2         103    Priya  21.444444  9.100000   96.000000           1       1
3         104    Rahul  20.000000  7.888889   75.000000           0       0
4         105    Sneha  23.000000  8.000000   87.777778           1       1

=== Unique values in 'placed' ===
[1 0] | dtype: int32

=== Unique values in 'internship' ===
[1 0] | dtype: int32

=== Target Distribution ===
placed
0    7
1    5
Name: count, dtype: int64


# Feature Selection

In [4]:
# Use .copy() to prevent SettingWithCopyWarning
result_df = df[["cgpa", "attendance", "internship"]].copy()

print("=== Feature Matrix (X) ===")
print(result_df)
print("\nDtypes:")
print(result_df.dtypes)

# corr() with parentheses — without () it just prints the method object
print("\n=== Correlation Matrix ===")
print(result_df.corr())

X = result_df
Y = df["placed"]

print("\n=== Target Variable (Y) ===")
print(Y.value_counts())

=== Feature Matrix (X) ===
        cgpa  attendance  internship
0   8.500000   92.000000           1
1   7.200000   85.000000           0
2   9.100000   96.000000           1
3   7.888889   75.000000           0
4   8.000000   87.777778           1
5   7.500000   88.000000           0
6   7.888889   91.000000           1
7   6.800000   87.777778           0
8   8.700000   89.000000           1
9   7.000000   79.000000           0
10  7.888889   95.000000           1
11  8.200000   87.777778           0

Dtypes:
cgpa          float64
attendance    float64
internship      int32
dtype: object

=== Correlation Matrix ===
                cgpa  attendance  internship
cgpa        1.000000    0.508924    0.694259
attendance  0.508924    1.000000    0.698164
internship  0.694259    0.698164    1.000000

=== Target Variable (Y) ===
placed
0    7
1    5
Name: count, dtype: int64


# Train / Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    train_size=0.7,
    random_state=10
)

print("Training set size :", X_train.shape)
print("Test set size     :", X_test.shape)
print("\nY_train values    :", Y_train.unique(), "| dtype:", Y_train.dtype)
print("Y_test  values    :", Y_test.unique(),  "| dtype:", Y_test.dtype)

Training set size : (8, 3)
Test set size     : (4, 3)

Y_train values    : [0 1] | dtype: int32
Y_test  values    : [1 0] | dtype: int32


# Model Training & Evaluation

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error
)

# Train model
model = LogisticRegression()
model.fit(X_train, Y_train)

# Predictions
Y_pred_train = model.predict(X_train)
Y_pred_test  = model.predict(X_test)

print("=== Predictions vs Actual ===")
print(f"{'Index':<8} {'Actual':<10} {'Predicted':<10}")
print("-" * 30)
for idx, (actual, pred) in enumerate(zip(Y_test, Y_pred_test)):
    match = "✓" if actual == pred else "✗"
    print(f"{idx:<8} {actual:<10} {pred:<10} {match}")

=== Predictions vs Actual ===
Index    Actual     Predicted 
------------------------------
0        1          1          ✓
1        0          1          ✗
2        0          0          ✓
3        0          1          ✗


# Error Metrics

In [7]:
# ── Accuracy ──────────────────────────────────────────────────────────────────
train_accuracy = accuracy_score(Y_train, Y_pred_train)
test_accuracy  = accuracy_score(Y_test,  Y_pred_test)

# ── Error Rate (Classification Error) ─────────────────────────────────────────
train_error_rate = 1 - train_accuracy
test_error_rate  = 1 - test_accuracy

# ── MAE — Mean Absolute Error ──────────────────────────────────────────────────
# How many predictions are wrong on average (0 = perfect, 1 = all wrong)
train_mae = mean_absolute_error(Y_train, Y_pred_train)
test_mae  = mean_absolute_error(Y_test,  Y_pred_test)

# ── MSE — Mean Squared Error ───────────────────────────────────────────────────
# Penalises wrong predictions more heavily
train_mse = mean_squared_error(Y_train, Y_pred_train)
test_mse  = mean_squared_error(Y_test,  Y_pred_test)

# ── RMSE — Root Mean Squared Error ────────────────────────────────────────────
train_rmse = np.sqrt(train_mse)
test_rmse  = np.sqrt(test_mse)

# ── Precision, Recall, F1 ─────────────────────────────────────────────────────
precision = precision_score(Y_test, Y_pred_test, zero_division=0)
recall    = recall_score(Y_test,    Y_pred_test, zero_division=0)
f1        = f1_score(Y_test,        Y_pred_test, zero_division=0)

# ── Print Summary ─────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════╗")
print("║          ERROR METRICS SUMMARY               ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  {'Metric':<22} {'Train':<10} {'Test':<10} ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  {'Accuracy':<22} {train_accuracy:<10.4f} {test_accuracy:<10.4f} ║")
print(f"║  {'Error Rate (1-Acc)':<22} {train_error_rate:<10.4f} {test_error_rate:<10.4f} ║")
print(f"║  {'MAE':<22} {train_mae:<10.4f} {test_mae:<10.4f} ║")
print(f"║  {'MSE':<22} {train_mse:<10.4f} {test_mse:<10.4f} ║")
print(f"║  {'RMSE':<22} {train_rmse:<10.4f} {test_rmse:<10.4f} ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  {'Precision (Test)':<22} {precision:<10.4f}            ║")
print(f"║  {'Recall (Test)':<22} {recall:<10.4f}            ║")
print(f"║  {'F1-Score (Test)':<22} {f1:<10.4f}            ║")
print("╚══════════════════════════════════════════════╝")

╔══════════════════════════════════════════════╗
║          ERROR METRICS SUMMARY               ║
╠══════════════════════════════════════════════╣
║  Metric                 Train      Test       ║
╠══════════════════════════════════════════════╣
║  Accuracy               1.0000     0.5000     ║
║  Error Rate (1-Acc)     0.0000     0.5000     ║
║  MAE                    0.0000     0.5000     ║
║  MSE                    0.0000     0.5000     ║
║  RMSE                   0.0000     0.7071     ║
╠══════════════════════════════════════════════╣
║  Precision (Test)       0.3333                ║
║  Recall (Test)          1.0000                ║
║  F1-Score (Test)        0.5000                ║
╚══════════════════════════════════════════════╝


# Confusion Matrix & Classification Report

In [8]:
cm = confusion_matrix(Y_test, Y_pred_test)

print("=== Confusion Matrix (Test Set) ===")
print(f"\n{'':>20} Predicted")
print(f"{'':>20}  0      1")
print(f"  Actual  0     {cm[0][0]}      {cm[0][1]}")
print(f"          1     {cm[1][0]}      {cm[1][1]}")
print()
print(f"  True Negatives  (TN) = {cm[0][0]}  — Correctly predicted NOT placed")
print(f"  False Positives (FP) = {cm[0][1]}  — Predicted placed, but actually NOT placed")
print(f"  False Negatives (FN) = {cm[1][0]}  — Predicted NOT placed, but actually placed")
print(f"  True Positives  (TP) = {cm[1][1]}  — Correctly predicted placed")

print("\n=== Full Classification Report ===")
print(classification_report(Y_test, Y_pred_test, target_names=["Not Placed (0)", "Placed (1)"]))

=== Confusion Matrix (Test Set) ===

                     Predicted
                      0      1
  Actual  0     1      2
          1     0      1

  True Negatives  (TN) = 1  — Correctly predicted NOT placed
  False Positives (FP) = 2  — Predicted placed, but actually NOT placed
  False Negatives (FN) = 0  — Predicted NOT placed, but actually placed
  True Positives  (TP) = 1  — Correctly predicted placed

=== Full Classification Report ===
                precision    recall  f1-score   support

Not Placed (0)       1.00      0.33      0.50         3
    Placed (1)       0.33      1.00      0.50         1

      accuracy                           0.50         4
     macro avg       0.67      0.67      0.50         4
  weighted avg       0.83      0.50      0.50         4

